# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [34]:
# Load the libraries as required.
%load_ext dotenv
%dotenv 
import os
import sys
sys.path.append(os.getenv('SRC_DIR'))
import pandas as pd
import numpy as np
import os

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, ParameterGrid
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [10]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [11]:
target = 'area'
categorical = ['month', 'day']
numeric = [c for c in fires_dt.columns if c not in categorical + [target]]

X = fires_dt[categorical + numeric]
y = fires_dt[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [21]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   
    ('scaler', StandardScaler())                
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),           
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preproc1 = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric),
        ('cat', categorical_pipeline, categorical),
    ],
    remainder='drop' 
)


### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [26]:


skews = X_train[numeric].skew(numeric_only=True)
skewed_num = skews.index[skews.abs() > 1].tolist()
rest_num = [c for c in numeric if c not in skewed_num]

num_skew_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer(method='yeo-johnson', standardize=True))])

num_rest_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', ohe)
])

preproc2 = ColumnTransformer(
    transformers=[
        ('num_skew', num_skew_pipe, skewed_num),
        ('num_rest', num_rest_pipe, rest_num),
        ('cat', cat_pipe, categorical),
    ],
    remainder='drop'
)

print("Skewed numeric transformed:", skewed_num)
print("Rest numeric scaled:", rest_num)


Skewed numeric transformed: ['ffmc', 'dc', 'rain']
Rest numeric scaled: ['coord_x', 'coord_y', 'dmc', 'isi', 'temp', 'rh', 'wind']


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [28]:
# Pipeline A = preproc1 + baseline
pipe_A = Pipeline([
    ("preprocessing", preproc1),
    ("regressor", Ridge(random_state=42))
])

In [29]:
# Pipeline B = preproc2 + baseline
pipe_B = Pipeline([
    ("preprocessing", preproc2),
    ("regressor", Ridge(random_state=42))
])

In [31]:
# Pipeline C = preproc1 + advanced model
pipe_C = Pipeline([
    ("preprocessing", preproc1),
    ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1))
])

In [30]:
# Pipeline D = preproc2 + advanced model
pipe_D = Pipeline([
    ("preprocessing", preproc2),
    ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1))
])
    

In [35]:
pipelines = {
    "A_preproc1_Ridge": pipe_A,
    "B_preproc2_Ridge": pipe_B,
    "C_preproc1_RF":    pipe_C,
    "D_preproc2_RF":    pipe_D,
}

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [33]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = "neg_root_mean_squared_error"

In [36]:
param_grids = {
    "A_preproc1_Ridge": {"regressor__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]},
    "B_preproc2_Ridge": {"regressor__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]},
    "C_preproc1_RF": {
        "regressor__n_estimators": [200, 500],
        "regressor__max_depth": [None, 10],
        "regressor__min_samples_leaf": [1, 3],
    },
    "D_preproc2_RF": {
        "regressor__n_estimators": [200, 500],
        "regressor__max_depth": [None, 10],
        "regressor__min_samples_leaf": [1, 3],
    },
}

In [38]:
def run_grid(name, pipe, grid):
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring=scoring,
        cv=cv,
        n_jobs=-1
    )
    gs.fit(X_train, y_train)
    cv_rmse = -gs.best_score_
    print(f"{name}: best CV RMSE = {cv_rmse:.3f} | params = {gs.best_params_}")
    return {
        "pipeline": name,
        "best_estimator": gs.best_estimator_,
        "best_params": gs.best_params_,
        "cv_rmse": cv_rmse,
    }

In [39]:
results = []
for name, pipe in pipelines.items():
    results.append(run_grid(name, pipe, param_grids[name]))

results_df = pd.DataFrame(results).sort_values("cv_rmse")
results_df

A_preproc1_Ridge: best CV RMSE = 39.709 | params = {'regressor__alpha': 1000.0}
B_preproc2_Ridge: best CV RMSE = 39.663 | params = {'regressor__alpha': 1000.0}
C_preproc1_RF: best CV RMSE = 43.009 | params = {'regressor__max_depth': 10, 'regressor__min_samples_leaf': 3, 'regressor__n_estimators': 500}
D_preproc2_RF: best CV RMSE = 43.022 | params = {'regressor__max_depth': None, 'regressor__min_samples_leaf': 3, 'regressor__n_estimators': 500}


,pipeline,best_estimator,best_params,cv_rmse
1,B_preproc2_Ridge,"(ColumnTransformer(transformers=[('num_skew',\...",{'regressor__alpha': 1000.0},39.662636
0,A_preproc1_Ridge,"(ColumnTransformer(transformers=[('num',\n ...",{'regressor__alpha': 1000.0},39.708750
2,C_preproc1_RF,"(ColumnTransformer(transformers=[('num',\n ...","{'regressor__max_depth': 10, 'regressor__min_s...",43.008949
3,D_preproc2_RF,"(ColumnTransformer(transformers=[('num_skew',\...","{'regressor__max_depth': None, 'regressor__min...",43.021880


# Evaluate

+ Which model has the best performance?

Ridge regressor model has the best performance with pipeline B
B_preproc2_Ridge	

# Export

+ Save the best performing model to a pickle file.

In [42]:
import joblib

best_model = results_df.iloc[0]["best_estimator"]
joblib.dump(best_model, "best_forestfire_model.pkl")


['best_forestfire_model.pkl']

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [43]:
best_row   = results_df.iloc[0]
best_model = best_row["best_estimator"]

preproc = best_model.named_steps["preprocessing"]
model   = best_model.named_steps["regressor"]

X_train_pre = preproc.transform(X_train)
X_test_pre  = preproc.transform(X_test)

try:
    feature_names = preproc.get_feature_names_out()
except Exception:
    feature_names = [f"feat_{i}" for i in range(X_train_pre.shape[1])]


In [45]:
import shap
import numpy as np


try:
    explainer = shap.LinearExplainer(model, X_train_pre)
    shap_values_test = explainer(X_test_pre)  
    sv_values = shap_values_test.values
    base_value = shap_values_test.base_values
except Exception:
    explainer = shap.LinearExplainer(model, X_train_pre)
    sv_values = explainer.shap_values(X_test_pre)
    base_value = explainer.expected_value

i = 0
print("Explaining test row:", i)


try:
    shap.plots.waterfall(shap.Explanation(values=sv_values[i],
                                          base_values=base_value,
                                          data=X_test_pre[i],
                                          feature_names=feature_names),
                         max_display=12)
except Exception:
    contrib = (np.abs(sv_values[i])).astype(float)
    top_idx = np.argsort(contrib)[::-1][:12]
    print("Top local feature contributions:")
    for j in top_idx:
        print(f"{feature_names[j]:25s}  SHAP={sv_values[i][j]: .4f}  value={X_test_pre[i][j]: .4f}")


idx = np.random.RandomState(42).choice(X_train_pre.shape[0], size=min(500, X_train_pre.shape[0]), replace=False)
shap_values_train = explainer(X_train_pre[idx]) if hasattr(explainer, "__call__") else explainer.shap_values(X_train_pre[idx])
sv_train = shap_values_train.values if hasattr(shap_values_train, "values") else shap_values_train

mean_abs = np.mean(np.abs(sv_train), axis=0)
global_imp = (pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs})
                .sort_values("mean_abs_shap", ascending=False)
                .reset_index(drop=True))

global_imp.head(15)


Explaining test row: 0
Top local feature contributions:
num_rest__rh               SHAP=-1.9179  value= 2.9841
num_rest__dmc              SHAP=-1.6096  value=-1.2806
num_rest__temp             SHAP=-1.1397  value=-1.3119
num_rest__coord_x          SHAP= 0.9063  value= 0.5843
num_skew__ffmc             SHAP=-0.7547  value=-1.6485
num_rest__coord_y          SHAP= 0.3705  value= 0.5615
num_skew__dc               SHAP=-0.3112  value=-1.7406
num_rest__wind             SHAP= 0.1748  value= 0.5069
cat__day_sat               SHAP= 0.1475  value= 1.0000
num_skew__rain             SHAP= 0.1373  value=-0.1214
num_rest__isi              SHAP=-0.1223  value=-1.3544
cat__day_fri               SHAP= 0.0682  value= 0.0000


,feature,mean_abs_shap
0,num_rest__coord_x,1.112071
1,num_rest__dmc,1.005873
2,num_rest__temp,0.765017
3,num_rest__coord_y,0.514630
4,num_rest__rh,0.501442
5,num_skew__ffmc,0.346931
6,num_rest__wind,0.249161
7,num_skew__rain,0.183215
8,num_skew__dc,0.172343
9,cat__day_fri,0.091509


Remove features with low SHAP value close to 0. I would test by retraining the same pipeline without the selected features to see a difference in RMSE. If it improves, drop the values, if not keep them. 

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.